### LLaVA use case demonstration

At that notebook you can see simple example of how to use TransformerLens for LLaVA interpretability. More specifically you can pass united image patch embeddings and textual embedding to LLaVA language model (Vicuna) with TransformerLens and get logits and cache that contains activations for next analysis. Here we consider the simplest example of LLaVA and TransformerLens sharing. 

In [1]:
# import staff
import sys

# Uncomment if use clonned version of TransformerLens
# currently forked version https://github.com/zazamrykh/TransformerLens supports
TL_path = r"../"
if TL_path not in sys.path:
    sys.path.insert(0, TL_path)
    sys.path.insert(0, TL_path + r"/transformer_lens")

import torch
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
)  # Should update transformer to latest version

# For image loading
from PIL import Image
import requests
from io import BytesIO


device = "cuda" if torch.cuda.is_available() else "cpu"

import matplotlib.pyplot as plt

%matplotlib inline

from transformer_lens import HookedTransformer
import circuitsvis as cv

_ = torch.set_grad_enabled(False)

Load llava model from hugging face. Load some revision because at this moment newest one is not working.

In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"

llava = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    load_in_4bit=False,
    low_cpu_mem_usage=True,
    revision="a272c74",
    device_map="cpu",
)

for param in llava.parameters():  # At this demo we don't need grads
    param.requires_grad = False

processor = AutoProcessor.from_pretrained(model_id, revision="a272c74")
tokenizer = processor.tokenizer

# Taking model apart
language_model = llava.language_model.eval()
config = language_model.config
print("Base language model:", config._name_or_path)

vision_tower = llava.vision_tower.to(device).eval()
projector = llava.multi_modal_projector.to(device).eval()

In [3]:
from transformers import LlavaForConditionalGeneration
from transformers.models.llava.processing_llava import LlavaProcessor
from transformers.models.clip.image_processing_clip import CLIPImageProcessor


def get_inputs(processor, image: Image, text: str, device="cuda"):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": text},
                {"type": "image"},
            ],
        },
    ]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(
        device, torch.float16
    )
    return inputs


# You can write your own version of getting language model's input embeddings similar way
# This function will not be working with old transformers library version. Should update transformers library.
def get_llm_input_embeddings(llava, processor, image: Image, text: str, device="cuda"):
    """Extract features from image, project them to LLM's space and insert them to text embedding sequence.
    Returns:
        inputs_embeds, attention_mask, labels, position_ids - input for language model of LLaVA
    """
    inputs = get_inputs(processor, image, text, device)

    llava.vision_tower.to(device)
    llava.multi_modal_projector.to(device)

    clip_output = llava.vision_tower(inputs["pixel_values"])
    projector_output = llava.multi_modal_projector(clip_output.last_hidden_state)

    before_device = llava.language_model.model.embed_tokens.weight.device
    llava.language_model.model.embed_tokens.to(device)
    text_embeddings = llava.language_model.model.embed_tokens(inputs["input_ids"])
    llava.language_model.model.embed_tokens.to(before_device)

    print("shape of projector_output", projector_output.shape)
    print("shape of text_embeddings", text_embeddings.shape)

    full_sequence = torch.hstack([projector_output, text_embeddings])

    attention_mask = torch.ones(
        full_sequence.shape[:-1], device=full_sequence.device, dtype=int
    )
    inputs_embeds, attention_mask, labels, position_ids = (
        llava._merge_input_ids_with_image_features(
            projector_output,
            text_embeddings,
            inputs["input_ids"],
            attention_mask,
            labels=None,
        )
    )  # Access to private member... Well, but what can i do :-)

    _, img_token_idxs = torch.where(
        inputs["input_ids"] == llava.config.image_token_index
    )
    fst_cls_token_idx = img_token_idxs.item()

    return inputs_embeds, attention_mask, labels, position_ids, fst_cls_token_idx

Okay, now create HookedTransformer model

In [ ]:
hooked_llm = HookedTransformer.from_pretrained(
    "llama-7b-hf",  # Use config of llama
    center_unembed=False,
    fold_ln=False,
    fold_value_biases=False,
    device="cuda",
    hf_model=language_model,  # Use Vicuna's weights
    tokenizer=tokenizer,
    center_writing_weights=False,
    dtype=torch.float16,
    vocab_size=language_model.config.vocab_size,  # New argument. llama and vicuna have different vocab size, so we pass it here
)

for param in hooked_llm.parameters():
    param.requires_grad = False

Now try if hooked model is working

In [ ]:
image_url = "https://github.com/zazamrykh/PicFinder/blob/main/images/doge.jpg?raw=true"


def load_image(url):
    response = requests.get(url)
    response.raise_for_status()
    return Image.open(BytesIO(response.content))


image = load_image(image_url)
plt.axis("off")
_ = plt.imshow(image)

In [ ]:
question = "What do you see on photo?"
inputs_embeds, attention_mask, labels, position_ids, fst_cls_token_idx = (
    get_llm_input_embeddings(llava, processor, image, question, device=device)
)

# Return tokens
outputs = hooked_llm.generate(
    inputs_embeds, max_new_tokens=30, do_sample=True, return_type="tokens"
)
generated_text = processor.decode(outputs[0], skip_special_tokens=True)
print("Generated text:", generated_text)

In [ ]:
# Now return embeddings and then project them on vocab space
outputs = hooked_llm.generate(
    inputs_embeds,
    max_new_tokens=30,
    do_sample=True,
)

logits = outputs[:, -30:, :].to(device) @ language_model.model.embed_tokens.weight.T.to(
    device
)
generated_text = processor.decode(logits.argmax(-1)[0], skip_special_tokens=True)
print("Generated text:", generated_text)

In [ ]:
processor.decode(
    (outputs @ language_model.model.embed_tokens.weight.T.to(device)).argmax(dim=-1)[0]
)

As we can see everything is working. Now try visualize attention patterns in generated output.

In [9]:
logits, cache = hooked_llm.run_with_cache(
    inputs_embeds, start_at_layer=0, remove_batch_dim=True
)

In [10]:
# Here we visualize attention for the last 30 tokens.
layer_to_visualize = 16
tokens_to_show = 30
attention_pattern = cache["pattern", layer_to_visualize, "attn"]

product = inputs_embeds @ language_model.model.embed_tokens.weight.T.to(
    device
)  # Project embeddings to vocab
llama_str_tokens = hooked_llm.to_str_tokens(product.argmax(dim=-1)[0])

As we can see image tokens also appears and can be used for multimodal attention exploration. 

In [ ]:
import numpy as np


def show_imgs_side_by_side(image1, image2):
    # Display
    plt.figure(figsize=(10, 5))

    # Original image
    plt.subplot(1, 2, 1)
    plt.imshow(image1)
    plt.title("Image 1")
    plt.axis("off")

    # Processed image
    plt.subplot(1, 2, 2)
    plt.imshow(image2)
    plt.title("Image 2")
    plt.axis("off")

    plt.show()


def get_processed_image(image, processor):
    # Process the image
    inputs = processor.image_processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"]

    # Convert the normalized tensor back to an image
    # Undo the normalization (usually mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
    mean = torch.tensor(processor.image_processor.image_mean).view(3, 1, 1)
    std = torch.tensor(processor.image_processor.image_std).view(3, 1, 1)

    # Denormalize
    img_denorm = pixel_values[0] * std + mean

    # Convert to numpy and transpose from (C, H, W) to (H, W, C)
    img_array = img_denorm.numpy().transpose(1, 2, 0)

    # Clip values to valid image range
    img_array = np.clip(img_array, 0, 1)
    return img_array


huskies_img = Image.open("../assets/huskies.jpg")
huskies_img_processed = get_processed_image(huskies_img, processor)
show_imgs_side_by_side(huskies_img, huskies_img_processed)


In [12]:
attn = attention_pattern[:12]
input_ids = get_inputs(processor, image, question).input_ids[0]
token_ids = torch.cat(
    [
        input_ids[:fst_cls_token_idx],
        torch.tensor(llava.config.image_token_index, device=device).repeat(24 * 24 + 1),
        input_ids[fst_cls_token_idx + 1 :],
    ]
)
str_tokens = processor.batch_decode(token_ids)
str_tokens[fst_cls_token_idx] = "[CLS]"

In [ ]:
from circuitsvis.attention import attention_heads

display(
    attention_heads(
        tokens=str_tokens,
        attention=attn.cpu().tolist(),
        max_value=attn.max().item() / 100,
        min_value=attn.min().item(),
        image=image_url,
        image_grid_dims=(24, 24),
        image_tokens_start=fst_cls_token_idx + 1,
    )
)